In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import shap
from lime.lime_tabular import LimeTabularExplainer
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference, equalized_odds_difference

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, precision_recall_curve, confusion_matrix, classification_report
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Set aesthetic defaults
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Environment initialized successfully.')

## 1. Data Ingestion & Data Quality Profiling
We load the consumer credit dataset and execute data validation rules.
Crucially, missing values in `person_emp_length` are preserved rather than dropped to allow median imputation downstream.

In [ ]:
data_path = '../data/credit_risk_dataset.csv'
df = pd.read_csv(data_path)
print(f'Raw dataset shape: {df.shape}')
display(df.head(3))

# Audit missingness
print('\nMissing values by column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Deduplicate and apply logical domain filters
df_clean = df.drop_duplicates().copy()
df_clean = df_clean[(df_clean['person_age'] >= 18) & (df_clean['person_age'] <= 85)]

# Preserve NaNs in employment length while removing impossible work experience
valid_emp = df_clean['person_emp_length'].isna() | (df_clean['person_emp_length'] <= (df_clean['person_age'] - 14))
df_clean = df_clean[valid_emp]
df_clean = df_clean[(df_clean['person_income'] > 0) & (df_clean['loan_amnt'] > 0)]

# Add sensitive attribute for fair lending audit
df_clean['age_group'] = np.where(df_clean['person_age'] < 25, 'Young (<25)', 'Mature (25+)')
print(f'\nValidated clean dataset shape: {df_clean.shape}')

## 2. Exploratory Data Analysis & Target Imbalance

In [ ]:
target_counts = df_clean['loan_status'].value_counts()
print(f'Target Distribution:\n{target_counts}')
print(f"Default Rate: {df_clean['loan_status'].mean():.2%}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(x=target_counts.index, y=target_counts.values, palette=['#0284c7', '#f43f5e'], ax=ax1)
ax1.set_title('Class Imbalance in Loan Status (0: Paid, 1: Default)')
ax1.set_xticklabels(['Non-Default', 'Default'])

sns.boxplot(data=df_clean, x='loan_grade', y='loan_percent_income', hue='loan_status', palette=['#10b981', '#f43f5e'], ax=ax2)
ax2.set_title('Loan-to-Income Ratio Across Credit Grades')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing Pipeline
We construct a Scikit-Learn `ColumnTransformer` to handle numerical and categorical features uniformly.

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])

X = df_clean[num_cols + cat_cols]
y = df_clean['loan_status'].astype(int)
sens = df_clean['age_group']

X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sens, test_size=0.20, random_state=42, stratify=y
)

neg, pos = np.bincount(y_train)
scale_weight = neg / pos
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}. Scale Pos Weight: {scale_weight:.2f}')

## 4. Multi-Model Benchmarking & MLflow Tracking
We train and compare 4 distinct machine learning algorithms with automated MLflow experiment tracking.

In [ ]:
mlflow.set_experiment('credit_risk_underwriting')

models = {
    'Logistic_Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random_Forest': RandomForestClassifier(n_estimators=150, class_weight='balanced', max_depth=10, random_state=42),
    'LightGBM': LGBMClassifier(scale_pos_weight=scale_weight, random_state=42, verbose=-1),
    'XGBoost': XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=42)
}

benchmark_rows = []
for name, clf in models.items():
    with mlflow.start_run(run_name=f'bench_{name}'):
        pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_test)[:, 1]
        preds = (proba >= 0.5).astype(int)
        
        row = {
            'Model': name,
            'ROC-AUC': roc_auc_score(y_test, proba),
            'F1': f1_score(y_test, preds),
            'Precision': precision_score(y_test, preds),
            'Recall': recall_score(y_test, preds),
            'Brier': brier_score_loss(y_test, proba)
        }
        mlflow.log_metrics(row)
        benchmark_rows.append(row)

df_bench = pd.DataFrame(benchmark_rows)
display(df_bench.sort_values('ROC-AUC', ascending=False))

## 5. Hyperparameter Tuning & Probability Calibration

In [ ]:
xgb_pipe = Pipeline([('prep', preprocessor), ('clf', XGBClassifier(scale_pos_weight=scale_weight, eval_metric='logloss', random_state=42))])
param_dist = {
    'clf__n_estimators': [150, 250, 350],
    'clf__max_depth': [4, 5, 6],
    'clf__learning_rate': [0.03, 0.08, 0.12],
    'clf__subsample': [0.8, 1.0],
    'clf__colsample_bytree': [0.8, 1.0]
}
search = RandomizedSearchCV(xgb_pipe, param_dist, n_iter=8, cv=4, scoring='average_precision', random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print(f'Best Hyperparameters: {search.best_params_}')

# Calibrate probabilities using Sigmoid / Platt Scaling
calibrated_model = CalibratedClassifierCV(estimator=best_model, method='sigmoid', cv=4)
calibrated_model.fit(X_train, y_train)
cal_proba = calibrated_model.predict_proba(X_test)[:, 1]
print(f'Calibrated ROC-AUC: {roc_auc_score(y_test, cal_proba):.4f} | Brier Loss: {brier_score_loss(y_test, cal_proba):.4f}')

## 6. Optimal Decision Threshold Tuning
We compute the Precision-Recall curve and identify the threshold maximizing the true harmonic mean $F_1 = 2 \times \frac{P \times R}{P + R}$.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, cal_proba)
f1_scores = np.divide(2 * (precisions * recalls), (precisions + recalls), out=np.zeros_like(precisions), where=(precisions + recalls) > 0)
best_idx = np.argmax(f1_scores[:-1])
optimal_threshold = float(thresholds[best_idx])
optimal_f1 = float(f1_scores[best_idx])
print(f'Optimal Decision Threshold: {optimal_threshold:.4f} (Max F1: {optimal_f1:.4f})')

final_preds = (cal_proba >= optimal_threshold).astype(int)
print('\nClassification Report at Optimal Threshold:')
print(classification_report(y_test, final_preds, target_names=['Non-Default', 'Default']))

## 7. Model Explainability: Global & Local SHAP
We extract the fitted tree booster to calculate exact Shapley values via `TreeExplainer`.

In [ ]:
fitted_prep = best_model.named_steps['prep']
raw_xgb = best_model.named_steps['clf']
feature_names = list(fitted_prep.get_feature_names_out())

X_test_trans = fitted_prep.transform(X_test)
X_test_df = pd.DataFrame(X_test_trans, columns=feature_names)

# Compatibility patch for modern XGBoost in SHAP
try:
    orig_decode = shap.explainers._tree.decode_ubjson_buffer
    def patched_decode(fd):
        jm = orig_decode(fd)
        try:
            bs = jm['learner']['learner_model_param']['base_score']
            if isinstance(bs, str):
                jm['learner']['learner_model_param']['base_score'] = float(bs.strip('[]'))
        except Exception:
            pass
        return jm
    shap.explainers._tree.decode_ubjson_buffer = patched_decode
except Exception:
    pass

explainer = shap.TreeExplainer(raw_xgb)
shap_values = explainer.shap_values(X_test_df)

# Global Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Global Feature Importance', fontsize=13)
plt.tight_layout()
plt.show()

# Local Waterfall Explanation for Sample Applicant
sample_i = 5
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_i],
        base_values=explainer.expected_value,
        data=X_test_df.iloc[sample_i],
        feature_names=feature_names
    )
)

## 8. Algorithmic Fairness Audit (Fairlearn)
We compute Demographic Parity Difference and Equalized Odds Difference across sensitive age cohorts.

In [ ]:
dp_diff = demographic_parity_difference(y_test, final_preds, sensitive_features=sens_test)
eo_diff = equalized_odds_difference(y_test, final_preds, sensitive_features=sens_test)
print(f'Demographic Parity Difference: {dp_diff:.4f} (Regulatory limit: <= 0.10)')
print(f'Equalized Odds Difference:     {eo_diff:.4f} (Regulatory limit: <= 0.10)')
status = 'COMPLIANT (PASS)' if dp_diff <= 0.10 and eo_diff <= 0.10 else 'REVIEW REQUIRED'
print(f'Fair Lending Compliance Status: {status}')